# 01 – Eksploracyjna Analiza Danych (EDA)
**Dataset:** Car Prices Poland (Kaggle)  
**Cel:** Zapoznanie się z danymi, analiza rozkładów, wykrywanie anomalii i wartości brakujących.

## 1. Import bibliotek i wczytanie danych

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 5)
from pathlib import Path
import os
PROJECT_ROOT = Path(os.getcwd())
# Jeśli CWD to notebooks/, cofnij się poziom wyżej
if not (PROJECT_ROOT / 'data').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_PATH    = str(PROJECT_ROOT / 'data' / 'raw' / 'Car_Prices_Poland_Kaggle.csv')
PROC_PATH    = str(PROJECT_ROOT / 'data' / 'processed') + os.sep
FIGURES_PATH = str(PROJECT_ROOT / 'reports' / 'figures') + os.sep
os.makedirs(PROC_PATH, exist_ok=True)
os.makedirs(FIGURES_PATH, exist_ok=True)
print('PROJECT_ROOT:', PROJECT_ROOT)
print('FIGURES_PATH:', FIGURES_PATH)

In [ ]:
df = pd.read_csv(DATA_PATH, index_col=0)
print(f'Kształt danych: {df.shape}')
df.head(10)

## 2. Podstawowe informacje o zbiorze danych

In [ ]:
df.info()

In [ ]:
df.describe(include='all').T

## 3. Brakujące wartości

In [ ]:
missing = df.isnull().sum().sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Brakujące': missing, '% brakujących': missing_pct})
print(missing_df[missing_df['Brakujące'] > 0])

## 4. Rozkłady zmiennych numerycznych

In [ ]:
num_cols = ['year', 'mileage', 'vol_engine', 'price']
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()
for i, col in enumerate(num_cols):
    axes[i].hist(df[col].dropna(), bins=50, edgecolor='black', color='steelblue', alpha=0.8)
    axes[i].set_title(f'Rozkład: {col}')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Liczba ogłoszeń')
plt.suptitle('Rozkłady zmiennych numerycznych', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(FIGURES_PATH + '01_distributions_numeric.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Rozkład ceny – transformacja logarytmiczna

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(df['price'].dropna(), bins=80, color='steelblue', edgecolor='black', alpha=0.8)
axes[0].set_title('Rozkład ceny (PLN)')
axes[0].set_xlabel('Cena (PLN)')
axes[1].hist(np.log1p(df['price'].dropna()), bins=80, color='darkorange', edgecolor='black', alpha=0.8)
axes[1].set_title('Rozkład log(cena + 1)')
axes[1].set_xlabel('log(Cena + 1)')
plt.tight_layout()
plt.savefig(FIGURES_PATH + '01_price_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Zmienne kategoryczne

In [ ]:
cat_cols = ['mark', 'fuel', 'province']
for col in cat_cols:
    print(f'\n--- {col} --- ({df[col].nunique()} unikalnych wartości)')
    print(df[col].value_counts().head(15))

In [ ]:
top_marks = df['mark'].value_counts().head(15)
plt.figure(figsize=(12, 5))
sns.barplot(x=top_marks.values, y=top_marks.index, palette='Blues_d')
plt.title('Top 15 marek samochodów (liczba ogłoszeń)')
plt.xlabel('Liczba ogłoszeń')
plt.tight_layout()
plt.savefig(FIGURES_PATH + '01_top_marks.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
fuel_counts = df['fuel'].value_counts()
plt.figure(figsize=(8, 5))
sns.barplot(x=fuel_counts.index, y=fuel_counts.values, palette='Set2')
plt.title('Rozkład rodzaju paliwa')
plt.xlabel('Rodzaj paliwa')
plt.ylabel('Liczba ogłoszeń')
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig(FIGURES_PATH + '01_fuel_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Cena według rodzaju paliwa

In [ ]:
plt.figure(figsize=(12, 6))
order = df.groupby('fuel')['price'].median().sort_values(ascending=False).index
sns.boxplot(data=df, x='fuel', y='price', order=order, palette='Set3')
plt.title('Rozkład ceny według rodzaju paliwa')
plt.xlabel('Rodzaj paliwa')
plt.ylabel('Cena (PLN)')
plt.yscale('log')
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig(FIGURES_PATH + '01_price_by_fuel.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Cena według roku produkcji

In [ ]:
year_price = df.groupby('year')['price'].median().reset_index()
plt.figure(figsize=(14, 5))
sns.lineplot(data=year_price, x='year', y='price', marker='o', color='steelblue')
plt.title('Mediana ceny według roku produkcji')
plt.xlabel('Rok produkcji')
plt.ylabel('Mediana ceny (PLN)')
plt.tight_layout()
plt.savefig(FIGURES_PATH + '01_price_by_year.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Cena vs. przebieg

In [ ]:
sample = df.dropna(subset=['mileage', 'price']).sample(min(5000, len(df)), random_state=42)
plt.figure(figsize=(10, 6))
plt.scatter(sample['mileage'], sample['price'], alpha=0.2, s=10, color='steelblue')
plt.title('Cena vs. Przebieg')
plt.xlabel('Przebieg (km)')
plt.ylabel('Cena (PLN)')
plt.yscale('log')
plt.tight_layout()
plt.savefig(FIGURES_PATH + '01_price_vs_mileage.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Heatmapa korelacji zmiennych numerycznych

In [ ]:
corr = df[num_cols].corr()
plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt='.3f', cmap='coolwarm', center=0,
            square=True, linewidths=0.5)
plt.title('Heatmapa korelacji (Pearson)')
plt.tight_layout()
plt.savefig(FIGURES_PATH + '01_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('\nKorelacje z ceną:')
print(corr['price'].sort_values(ascending=False))

## 11. Top 15 marek – mediana ceny

In [ ]:
top15 = df['mark'].value_counts().head(15).index
mark_price = (df[df['mark'].isin(top15)]
              .groupby('mark')['price']
              .median()
              .sort_values(ascending=False))
plt.figure(figsize=(12, 5))
sns.barplot(x=mark_price.index, y=mark_price.values, palette='viridis')
plt.title('Mediana ceny dla Top 15 marek')
plt.xlabel('Marka')
plt.ylabel('Mediana ceny (PLN)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(FIGURES_PATH + '01_median_price_by_mark.png', dpi=150, bbox_inches='tight')
plt.show()

## 12. Cena według województwa

In [ ]:
province_price = (df.groupby('province')['price']
                  .median()
                  .sort_values(ascending=False)
                  .reset_index())
plt.figure(figsize=(14, 5))
sns.barplot(data=province_price, x='province', y='price', palette='coolwarm')
plt.title('Mediana ceny według województwa')
plt.xlabel('Województwo')
plt.ylabel('Mediana ceny (PLN)')
plt.xticks(rotation=60, ha='right')
plt.tight_layout()
plt.savefig(FIGURES_PATH + '01_price_by_province.png', dpi=150, bbox_inches='tight')
plt.show()

## 13. Wnioski z EDA

- Zbiór danych zawiera ogłoszenia sprzedaży samochodów z całej Polski (styczeń 2022).
- Cena jest **prawostronnie skośna** – transformacja `log` przybliża ją do rozkładu normalnego.
- **Rok produkcji** wykazuje silną dodatnią korelację z ceną; **przebieg** – ujemną.
- Pojemność silnika (`vol_engine`) ma umiarkowaną korelację z ceną.
- Samochody elektryczne i hybrydowe mają wyraźnie wyższe ceny mediany.
- Dane wymagają czyszczenia – patrz notebook `02_preprocessing.ipynb`.